[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P15/P15-1_Dispersion_CORRIGE.ipynb)

# Modéliser une dispersion

Chapitre 15 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Vos organes ne durent pas tous autant, et vos pièces usinées ne tombent pas toutes à la même cote. La moyenne ne dit ni quand arrivent les premières défaillances, ni si la production reste dans sa tolérance. Vous voudriez résumer l'étalement par une loi, juger si un écart est réel ou dû au hasard, et annoncer un résultat avec son incertitude.

**Objectif terminal.** Ce chapitre montre comment, avec le module `scipy.stats` de SciPy.

**Objectif instrumental.**

- Résumer une dispersion par la médiane, les quartiles et l'écart-type, et ajuster une droite avec son incertitude.
- Ajuster une loi, normale ou de Weibull, à des durées de vie et en lire un quantile.
- Juger un écart par un test et lire une valeur $p$ sans la sur-interpréter.
- Chiffrer l'incertitude d'une moyenne par un intervalle ou par le bootstrap, et calculer les indices $C_p$ et $C_{pk}$.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 7 et 8, après une filière scientifique.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-RUL.xlsx` | 240 | 6 | couple en N·m, puissance en kW, masse en kg, temps de réponse en ms, durée de vie résiduelle en jours |
| `MecaIA-TimeSeries_RUL.xlsx` | 18855 | 10 | cycles en heures, charge en %, vitesse en tr/min, température en K, pression en bar, vibration en mm/s, courant en A, contamination en ppm |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 15.1 | La durée de vie baisse-t-elle avec la masse vibrante ? | 20 min |
| 15.2 | Weibull d'une flotte, et son B10 | 20 min |
| 15.3 | Le test de Kolmogorov-Smirnov sur la flotte | 15 min |
| 15.4 | Cp et Cpk d'une cote usinée, avec leur incertitude (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [1]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-RUL.xlsx",
    "MecaIA-TimeSeries_RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-TimeSeries_RUL.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

jeux lus dans le dossier des données : MecaIA-RUL.xlsx, MecaIA-TimeSeries_RUL.xlsx


### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [2]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("pandas", version("pandas"))
print("scipy", version("scipy"))

Python 3.12.0
numpy 2.4.6
pandas 3.0.5
scipy 1.18.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule.

In [3]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [4]:
import math
import numpy as np
import pandas as pd
from scipy import stats
CHEMIN_RUL = DONNEES / "MecaIA-RUL.xlsx"
CHEMIN_FLOTTE = DONNEES / "MecaIA-TimeSeries_RUL.xlsx"
UNITE_MASSE = "kg"                    # lue dans la fiche : kg
UNITE_VIE_RUL = "jours"                   # jours
UNITE_VIE_FLOTTE = "heures"    # la fiche compte les cycles en heures

### Section 15.1 : la dispersion et la droite ajustée résument des mesures.

In [5]:
data = pd.read_excel(CHEMIN_RUL)
df = data.copy()
masse = df["masse_vibrante_kg"]
rul = df["RUL_jours"]
print(f"{len(df)} organes ; manquants dans les deux colonnes utiles : {int(masse.isna().sum() + rul.isna().sum())}")
assert (
    df.shape == (240, 6) and masse.isna().sum() + rul.isna().sum() == 0
), "240 lignes complètes pour les deux colonnes"

240 organes ; manquants dans les deux colonnes utiles : 0


In [6]:
# notion: quantiles
mediane = float(rul.median())
q1, q3 = float(rul.quantile(0.25)), float(rul.quantile(0.75))
print(f"durée de vie : médiane {mediane:.1f} {'jours'} ; quartiles {q1:.1f} et {q3:.1f} ; moyenne {rul.mean():.1f}")
assert mediane == np.median(rul.to_numpy()), "pandas et NumPy donnent la même médiane"
assert abs(q1 - np.quantile(rul.to_numpy(), 0.25)) < 1e-12, "même premier quartile"
assert q1 < mediane < q3 and abs(mediane - 224.5) < 0.05, "médiane attendue : 224,5 jours"
assert (
    mediane > rul.mean()
), "la médiane dépasse la moyenne : la distribution s'étire vers les courtes durées"

durée de vie : médiane 224.5 jours ; quartiles 136.5 et 262.2 ; moyenne 199.8


In [7]:
# notion: correlation
correlation = float(masse.corr(rul))
matrice = df.corr(numeric_only=True)                  # la colonne de texte est écartée
print(f"corrélation masse et durée de vie : {correlation:.3f} ; matrice : {matrice.shape}")
assert abs(correlation - (-0.552)) < 0.0005, "corrélation attendue : -0,552"
assert (
    abs(matrice.loc["masse_vibrante_kg", "RUL_jours"] - correlation) < 1e-12
), "même valeur dans la matrice"
assert (
    np.allclose(matrice.to_numpy(), matrice.to_numpy().T)
), "une matrice de corrélation est symétrique"

corrélation masse et durée de vie : -0.552 ; matrice : (5, 5)


In [8]:
# notion: linregress
ajustement = stats.linregress(masse, rul)             # pente, ordonnée, r, valeur p, incertitude de la pente
pente, ordonnee = np.polyfit(masse, rul, 1)
print(f"pente {ajustement.slope:.4f} ± {ajustement.stderr:.4f} {'jours'} par {'kg'} ; r = {ajustement.rvalue:.3f} ; ordonnée {ajustement.intercept:.1f}")
assert (
    abs(ajustement.slope - pente) < 1e-12 and abs(ajustement.intercept - ordonnee) < 1e-9
), "même droite que polyfit"
assert abs(ajustement.rvalue - correlation) < 1e-12, "r est la corrélation"
assert ajustement.pvalue < 1e-18, "une pente nulle est exclue"

pente -0.1260 ± 0.0123 jours par kg ; r = -0.552 ; ordonnée 372.3


## Exercice 15.1 : La durée de vie baisse-t-elle avec la masse vibrante ? (20 min)

Sur les 240 organes du jeu, ajustez la durée de vie en fonction de la masse vibrante avec `linregress`. Donnez la pente avec un intervalle d'environ $95\,\%$ ($1{,}96$ fois son incertitude), la perte de durée de vie pour $100$ kg de plus, et l'ordonnée à l'origine.

**Résultat attendu.** Pente de $-0{,}126$ jour par kg, avec une demi-largeur de 0,024 ; 12,6 jours de moins pour cent kilogrammes de plus ; ordonnée de 372,3 jours.

**Seconde voie.** `np.polyfit(masse, rul, 1)` redonne la pente $-0{,}1260$ ; la pente est aussi $r\,s_y/s_x$ avec $r = -0{,}552$.

**Indication.** Les deux colonnes sont complètes : les 240 organes servent. `masse = df["masse_vibrante_kg"]`, puis `ajust = stats.linregress(masse, rul)` ; lisez `ajust.slope`, `ajust.stderr` (la demi-largeur est `1.96 * ajust.stderr`) et `ajust.intercept`. La perte pour cent kilogrammes est `abs(100 * ajust.slope)`. La pente vaut plus de cinq fois son incertitude : une pente nulle est exclue.

In [9]:
# Solution :
demi_largeur = 1.96 * ajustement.stderr                  # intervalle d'environ 95 %
perte_cent_kg = abs(100 * ajustement.slope)
print(f"pente : {ajustement.slope:.3f} ± {demi_largeur:.3f} {'jours'} par {'kg'} ; {perte_cent_kg:.1f} {'jours'} de moins pour 100 {'kg'} de plus")
# Fin de la solution
assert abs(ajustement.slope - (-0.126)) < 0.0005, "pente attendue : -0,126 jour par kg"
assert abs(ordonnee - 372.3) < 0.05, "ordonnée attendue : 372,3 jours"
assert abs(ajustement.slope) > 5 * ajustement.stderr, "la pente est cinq fois son incertitude"
assert (
    abs(ajustement.slope - ajustement.rvalue * rul.std() / masse.std()) < 1e-12
), "pente = r fois le rapport des écarts-types"
print("Exercice 15.1 : contrôles passés")

pente : -0.126 ± 0.024 jours par kg ; 12.6 jours de moins pour 100 kg de plus
Exercice 15.1 : contrôles passés


### Section 15.2 : une loi décrit la dispersion.

In [10]:
# notion: lois
loi_normale = stats.norm(loc=0.0, scale=1.0)
print(f"norm : quantile à 97,5 % = {loi_normale.ppf(0.975):.4f} ; fonction de répartition en 0 : {loi_normale.cdf(0.0):.1f}")
assert abs(loi_normale.ppf(0.975) - 1.959964) < 1e-6 and loi_normale.cdf(0.0) == 0.5
mu_rul, sigma_rul = stats.norm.fit(rul)               # ajuste la moyenne et l'écart-type (maximum de vraisemblance)
assert (
    abs(mu_rul - rul.mean()) < 1e-9 and abs(sigma_rul - rul.std(ddof=0)) < 1e-9
), "norm.fit : ddof = 0"
print(f"loi normale ajustée sur la durée de vie : moyenne {mu_rul:.1f}, écart-type {sigma_rul:.1f} {'jours'}")

norm : quantile à 97,5 % = 1.9600 ; fonction de répartition en 0 : 0.5
loi normale ajustée sur la durée de vie : moyenne 199.8, écart-type 89.3 jours


In [11]:
# notion: weibull
flotte = pd.read_excel(CHEMIN_FLOTTE)
vie = flotte.groupby("ID_Machine")["Cycle_Operation_Heures"].max()      # durée de vie de chaque machine : son dernier cycle
print(f"flotte : {len(vie)} machines, durées de {int(vie.min())} à {int(vie.max())} {'heures'}")
assert flotte.shape == (18855, 10) and len(vie) == 100, "100 machines, 18 855 relevés"
assert int(vie.min()) == 50 and int(vie.max()) == 295, "durées de 50 à 295"
beta, _, eta = stats.weibull_min.fit(vie, floc=0)     # forme beta, position fixée à 0, échelle eta
loi_vie = stats.weibull_min(beta, loc=0, scale=eta)
b10 = float(loi_vie.ppf(0.10))                        # durée atteinte par 90 % des machines
# Droite de Weibull : ln(-ln(1 - F)) en fonction de ln(t) est une droite de pente beta (rangs médians de Benard).
t_tries = np.sort(vie.to_numpy())
rangs = (np.arange(1, len(t_tries) + 1) - 0.3) / (len(t_tries) + 0.4)
droite = stats.linregress(np.log(t_tries), np.log(-np.log(1 - rangs)))
eta_droite = math.exp(-droite.intercept / droite.slope)
print(f"Weibull par vraisemblance : beta {beta:.2f}, eta {eta:.1f}, B10 {b10:.1f} ; par la droite : beta {droite.slope:.2f}, eta {eta_droite:.1f}")

flotte : 100 machines, durées de 50 à 295 heures
Weibull par vraisemblance : beta 2.96, eta 211.7, B10 99.1 ; par la droite : beta 2.56, eta 213.8


In [12]:
assert (
    abs(loi_vie.cdf(eta) - (1 - math.exp(-1))) < 1e-12
), "63,2 % des machines ont cassé à t = eta"
assert (
    abs(droite.slope - beta) < 0.5 and abs(eta_droite - eta) < 5.0
), "la droite et la vraisemblance concordent à 15 %"
assert droite.rvalue > 0.95, "les points de Weibull sont presque alignés"

## Exercice 15.2 : Weibull d'une flotte, et son B10 (20 min)

Ajustez une loi de Weibull aux $100$ durées de vie de la flotte, avec la position fixée à zéro. Donnez $\beta$, $\eta$ et le B10. Combien de machines ont une durée de vie inférieure au B10, et combien en attendait-on ?

**Résultat attendu.** $\beta = 2{,}96$ ; $\eta = 211{,}7$ heures ; B10 de 99,1 heures ; 17 machines sur 100 sous le B10, pour dix attendues.

**Seconde voie.** La formule fermée $\eta\,(-\ln 0{,}9)^{1/\beta}$ redonne le B10 ; la droite de Weibull donne $\beta = 2{,}56$ et $\eta = 213{,}8$ heures, moins fiables que la vraisemblance.

**Indication.** `beta, _, eta = stats.weibull_min.fit(vie, floc=0)`, puis la méthode `ppf` de la loi figée, au quantile $0{,}10$, donne le B10 ; comparer les durées au B10, puis appeler `sum`, compte les machines dessous. Dix sur cent sont attendues sous le B10 par définition : en trouver dix-sept dit que la queue des courtes durées est plus lourde que celle de la loi ajustée.

Les contrôles de la cellule lisent la variable `sous_b10` : gardez ce nom.

In [13]:
# Solution :
print(f"beta = {beta:.2f} ; eta = {eta:.1f} {'heures'} ; B10 = {b10:.1f} {'heures'}")
# Fin de la solution
assert abs(beta - 2.96) < 0.005, "beta attendu : 2,96"
assert abs(eta - 211.7) < 0.05, "eta attendu : 211,7"
assert abs(b10 - 99.1) < 0.05, "B10 attendu : 99,1"
assert abs(b10 - eta * (-math.log(0.9))**(1 / beta)) < 1e-9, "formule fermée du B10"
# Solution :
sous_b10 = int((vie < b10).sum())
print(f"machines dont la durée de vie est inférieure au B10 : {sous_b10} sur {len(vie)}")
# Fin de la solution
assert (
    sous_b10 == 17
), "17 machines sur 100 sous le B10, pour 10 attendues : la queue des courtes durées est plus lourde"
print("Exercice 15.2 : contrôles passés")

beta = 2.96 ; eta = 211.7 heures ; B10 = 99.1 heures
machines dont la durée de vie est inférieure au B10 : 17 sur 100
Exercice 15.2 : contrôles passés


### Section 15.3 : un test juge un écart.

In [14]:
# notion: intervalles
moyenne_vie = float(vie.mean())
erreur_standard = float(vie.std(ddof=1) / math.sqrt(len(vie)))
bas, haut = stats.t.interval(0.95, df=len(vie) - 1, loc=moyenne_vie, scale=erreur_standard)
print(f"durée de vie moyenne : {moyenne_vie:.1f} {'heures'}, intervalle à 95 % : [{bas:.1f} ; {haut:.1f}]")
assert (
    bas < moyenne_vie < haut and abs((haut - bas) / 2 - 1.984 * erreur_standard) < 0.01
), "demi-largeur : 1,984 écart-types"

durée de vie moyenne : 188.6 heures, intervalle à 95 % : [174.2 ; 202.9]


In [15]:
# notion: tests
niveaux = df["niveau_sollicitation_thermique_cat"]
faible = rul[niveaux == "Faible"]
eleve = rul[niveaux == "Élevé"]
essai = stats.ttest_ind(faible, eleve, equal_var=False)       # comparaison de deux moyennes (Welch)
normalite = stats.shapiro(vie)
print("Student de Welch, durée de vie faible contre élevée : p sous 1e-10 :", essai.pvalue < 1e-10, f"; Shapiro sur la flotte : p = {normalite.pvalue:.4f}")
assert (
    essai.pvalue < 1e-10
), "les deux niveaux de sollicitation donnent des durées de vie très différentes"
assert normalite.pvalue < 0.01, "la normalité est rejetée : durées bornées entre 50 et 295 heures"
assert len(faible) == 119 and len(eleve) == 69, "effectifs des deux niveaux"

Student de Welch, durée de vie faible contre élevée : p sous 1e-10 : True ; Shapiro sur la flotte : p = 0.0001


## Exercice 15.3 : Le test de Kolmogorov-Smirnov sur la flotte (15 min)

Testez la compatibilité des durées de vie de la flotte avec la loi de Weibull ajustée à l'exercice précédent, puis avec une loi normale de même moyenne et de même écart-type. Que concluez-vous, sachant que les paramètres viennent des mêmes données ?

**Résultat attendu.** Weibull : $D = 0{,}114$, $p = 0{,}14$ ; loi normale : $p = 0{,}16$ ; les valeurs p ordinaires de Kolmogorov-Smirnov ne permettent pas de décider au seuil de $5\,\%$, les paramètres étant ajustés sur ces données.

**Seconde voie.** Le test de Shapiro-Wilk sur la flotte donne $p = 0{,}0001$ : il rejette la normalité, ce que les valeurs p de Kolmogorov-Smirnov ne permettent pas de décider ici.

**Indication.** `stats.kstest(vie, "weibull_min", args=(beta, 0, eta))` pour Weibull ; pour la loi normale, passez la fonction de répartition : `stats.kstest(vie, stats.norm(vie.mean(), vie.std()).cdf)`. Les deux valeurs p dépassent $0{,}05$, mais elles ne se lisent pas au seuil ordinaire : les paramètres ont été estimés sur les mêmes données, ce qui rend Kolmogorov-Smirnov trop indulgent et sa valeur p optimiste. Shapiro-Wilk, plus puissant, rejette la normalité. Ne pas rejeter ne démontre pas qu'une loi convient.

Les contrôles de la cellule lisent les variables `ks`, `ks_normale` : gardez ces noms.

In [16]:
# Solution :
ks = stats.kstest(vie, "weibull_min", args=(beta, 0, eta))
ks_normale = stats.kstest(vie, stats.norm(moyenne_vie, float(vie.std(ddof=1))).cdf)       # fonction de répartition en argument
print(f"Weibull : D = {ks.statistic:.3f}, p = {ks.pvalue:.2f} ; loi normale : p = {ks_normale.pvalue:.2f}")
# Fin de la solution
assert abs(ks.pvalue - 0.14) < 0.005, "valeur p attendue : 0,14"
assert (
    ks.pvalue > 0.05 and ks_normale.pvalue > 0.05
), "ni Weibull ni la loi normale ne sont rejetées au seuil de 5 %"
assert abs(ks.statistic - 0.114) < 0.0005, "écart maximal des répartitions : 0,114"
print("Exercice 15.3 : contrôles passés")

Weibull : D = 0.114, p = 0.14 ; loi normale : p = 0.16
Exercice 15.3 : contrôles passés


### Section 15.4 : le bootstrap et les indices Cp et Cpk.

In [17]:
# notion: bootstrap
boot_vie = stats.bootstrap((vie.to_numpy(),), np.mean, n_resamples=2000, confidence_level=0.95,
                           method="percentile", rng=np.random.default_rng(2026))
bas_b, haut_b = float(boot_vie.confidence_interval.low), float(boot_vie.confidence_interval.high)
print(f"bootstrap de la durée de vie moyenne : [{bas_b:.1f} ; {haut_b:.1f}] contre [{bas:.1f} ; {haut:.1f}] par la loi de Student")
assert (
    bas_b < moyenne_vie < haut_b and abs(bas_b - bas) < 2.5 and abs(haut_b - haut) < 2.5
), "les deux intervalles concordent"

bootstrap de la durée de vie moyenne : [174.7 ; 202.4] contre [174.2 ; 202.9] par la loi de Student


In [18]:
# notion: cp-cpk
# Une cote usinée de 50,000 mm, tolérance de plus ou moins 0,050 mm : 100 pièces simulées (graine fixée), centrées un
# peu trop haut, avec un écart-type de 0,012 mm.
rng = np.random.default_rng(2026)
cote_nominale, tolerance_mm = 50.000, 0.050
cotes = rng.normal(50.008, 0.012, 100)
limite_basse, limite_haute = cote_nominale - tolerance_mm, cote_nominale + tolerance_mm

def indices_capabilite(x):
    """Cp et Cpk d'un échantillon de cotes : Cp compare la tolérance à 6 sigma, Cpk tient compte du décentrage."""
    moyenne, sigma = np.mean(x), np.std(x, ddof=1)
    cp = (limite_haute - limite_basse) / (6 * sigma)
    cpk = min(limite_haute - moyenne, moyenne - limite_basse) / (3 * sigma)
    return cp, cpk

cp, cpk = indices_capabilite(cotes)
print(f"cote moyenne {cotes.mean():.4f} mm, écart-type {cotes.std(ddof=1):.4f} mm ; Cp {cp:.2f} ; Cpk {cpk:.2f}")
assert cpk <= cp, "Cpk ne dépasse jamais Cp : le décentrage ne peut que dégrader"

cote moyenne 50.0085 mm, écart-type 0.0123 mm ; Cp 1.36 ; Cpk 1.13


In [19]:
assert abs(cp - (limite_haute - limite_basse) / (6 * cotes.std(ddof=1))) < 1e-12

## Exercice 15.4 : Cp et Cpk d'une cote usinée, avec leur incertitude (défi, 25 min)

Une cote de $50{,}000\pm 0{,}050$ mm est mesurée sur cent pièces simulées : `rng = np.random.default_rng(2026)` puis `cotes = rng.normal(50.008, 0.012, 100)`. Calculez $C_p$ et $C_{pk}$, l'intervalle à $95\,\%$ de $C_{pk}$ par `stats.bootstrap` ($2000$ rééchantillons, méthode `"percentile"`, `rng=np.random.default_rng(2026)`), et la part des pièces hors tolérance prévue par la loi normale de même moyenne et de même écart-type que l'échantillon (`ddof=1`), en parties par million.

**Résultat attendu.** Moyenne de 50,0085 mm, écart-type de 0,0123 mm ; $C_p = 1{,}36$ ; $C_{pk} = 1{,}13$, avec un intervalle de 0,96 à 1,36 ; 370 pièces par million hors tolérance.

**Seconde voie.** Avec $s = 0{,}0123$ et la tolérance de $\pm0{,}050$ mm, $C_p = 2\times0{,}050/(6\,s)$ redonne 1,36.

**Indication.** Réutilisez la fonction `indices` avec les limites $49{,}950$ et $50{,}050$. Pour le bootstrap, passez `lambda x: indices(x, 49.95, 50.05)[1]` comme statistique. La part hors tolérance est la somme de `stats.norm.cdf(49.95, moyenne, s)` et de `stats.norm.sf(50.05, moyenne, s)`, multipliée par $10^6$. Le $C_{pk}$ estimé est bien au-dessus de $1$, mais son intervalle descend sous $1$ : cent pièces ne suffisent pas à garantir un indice.

Les contrôles de la cellule lisent les variables `bas_cpk`, `haut_cpk`, `hors_tolerance` : gardez ces noms.

In [20]:
# Solution :
boot_cpk = stats.bootstrap((cotes,), lambda x: indices_capabilite(x)[1], n_resamples=2000, confidence_level=0.95,
                           method="percentile", rng=np.random.default_rng(2026))
bas_cpk, haut_cpk = float(boot_cpk.confidence_interval.low), float(boot_cpk.confidence_interval.high)
hors_tolerance = float(stats.norm.cdf(limite_basse, cotes.mean(), cotes.std(ddof=1))
                       + stats.norm.sf(limite_haute, cotes.mean(), cotes.std(ddof=1)))
print(f"Cp {cp:.2f} ; Cpk {cpk:.2f}, intervalle à 95 % [{bas_cpk:.2f} ; {haut_cpk:.2f}] ; hors tolérance (loi normale) : {1e6 * hors_tolerance:.0f} ppm")
# Fin de la solution
assert bas_cpk < cpk < haut_cpk, "l'intervalle contient l'estimation"
assert abs(cp - 1.36) < 0.005 and abs(cpk - 1.13) < 0.005, "indices attendus : Cp 1,36 ; Cpk 1,13"
assert 1e-4 < hors_tolerance < 1e-3, "quelques pièces sur dix mille sortent de la tolérance"
print("Exercice 15.4 : contrôles passés")

Cp 1.36 ; Cpk 1.13, intervalle à 95 % [0.96 ; 1.36] ; hors tolérance (loi normale) : 370 ppm
Exercice 15.4 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F15-1 : Lois de probabilité (`scipy.stats.norm`)

In [21]:
import numpy as np
from scipy import stats

mu, sigma = 20.0, 0.05                  # moyenne et écart-type, mm
loi_cote = stats.norm(loc=mu, scale=sigma)   # diamètres
prob_inf = loi_cote.cdf(19.88)          # part sous la limite basse
prob_sup = loi_cote.sf(20.12)           # part dépassant la limite haute
rebut = prob_inf + prob_sup             # proportion hors tolérance
d_99 = loi_cote.ppf(0.99)               # taille dépassant 99 % du lot
texte = f"Rebut : {rebut * 100:.2f} % ; diamètre à 99 % : {d_99:.2f} mm"
print(texte.replace(".", ","))
assert np.isclose(loi_cote.cdf(mu), 0.5)   # la moyenne sépare la population

Rebut : 1,64 % ; diamètre à 99 % : 20,12 mm
